# Frequentist Hypothesis Testing: Z-Tests and T-Tests

## Overview

In this notebook, we use traditional statistical tests to determine whether email campaigns actually worked.
Instead of running every possible test and hoping for significance, we follow the **Primary → Secondary → Exploratory** framework.
This is the industry standard in A/B testing to control false positive rates and maintain scientific rigor.

## The Framework

### Primary Comparison (Strictest Standard)
- **One** pre-specified business question that we most need answered
- Significance level: **α = 0.05** (5% false positive rate)
- Example: "Did email campaigns increase conversion overall?"

### Secondary Comparisons (Planned Follow-ups)
- **Pre-planned tests** that drill into the primary result
- Significance level: **α = 0.05 / number of tests** (Bonferroni correction)
- Prevents "p-hacking" by adjusting for multiple testing
- Example: "Which email type (Mens vs Womens) performed better?"

### Exploratory Analysis (Hypothesis-Generating)
- Tests we run to **generate ideas** for future experiments
- No statistical correction (but findings require confirmation)
- Clearly labeled as exploratory
- Example: "Does purchase-history alignment matter?"

## Why This Matters

Without discipline, you could run 100 tests and expect ~5 to be "significant" by pure chance.
The Primary → Secondary → Exploratory structure:
- Shows the business we have a real hypothesis
- Controls false positive rate mathematically
- Separates "ideas worth testing" from "findings we're confident in"
- Makes results reproducible and defensible

In [1]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm, t as t_dist, ttest_ind, mannwhitneyu, levene, shapiro
import warnings
warnings.filterwarnings('ignore')

try:
    import plotly.graph_objects as go
    import plotly.express as px
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False
    print("Plotly not available; using matplotlib for all visualizations")

# Set plot style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Create output directory
output_dir = '../data/outputs/nb02'
os.makedirs(output_dir, exist_ok=True)
print(f"Output directory: {output_dir}")

Output directory: ../data/outputs/nb02


In [2]:
# Load the clean data from nb01
df = pd.read_csv('../data/outputs/nb01/nb01_hillstrom_clean.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
print(df.head())

# Check key columns
print(f"\n=== Verification ===")
print(f"'treatment' column exists: {'treatment' in df.columns}")
print(f"'email_match_simple' column exists: {'email_match_simple' in df.columns}")
print(f"\nSegment values in 'segment' column:")
print(df['segment'].value_counts())
print(f"\nTreatment distribution (0=Control, 1=Any Email):")
print(df['treatment'].value_counts())
print(f"\nEmail match distribution:")
print(df['email_match_simple'].value_counts())

Dataset shape: (64000, 24)

Columns: ['recency', 'history_segment', 'history', 'mens', 'womens', 'zip_code', 'newbie', 'channel', 'segment', 'visit', 'conversion', 'spend', 'email_match', 'email_match_simple', 'treatment', 'buyer_type', 'cross_shopper', 'zip_code_encoded', 'channel_encoded', 'history_log', 'spending_velocity', 'high_value', 'recency_segment', 'rfm_score']

First few rows:
   recency history_segment  history  mens  womens   zip_code  newbie channel  \
0       10  2) $100 - $200   142.44     1       0  Surburban       0   Phone   
1        6  3) $200 - $350   329.08     1       1      Rural       1     Web   
2        7  2) $100 - $200   180.65     0       1  Surburban       1     Web   
3        9  5) $500 - $750   675.83     1       0      Rural       1     Web   
4        2    1) $0 - $100    45.34     1       0      Urban       0     Web   

         segment  visit  ...  treatment   buyer_type cross_shopper  \
0  Womens E-Mail      0  ...          1    mens_only     

## Pre-Specified Analysis Plan

Before we run any tests, we formally commit to this analysis plan.
This prevents "p-hacking" and shows good scientific practice.

### PRIMARY COMPARISON (1 test)
**Question:** Did any email campaign increase conversions overall?

- **H₀ (Null):** Conversion rate (Any Email) = Conversion rate (No Email)
- **H₁ (Alternative):** Conversion rate (Any Email) ≠ Conversion rate (No Email)
- **Significance level:** α = 0.05
- **Test:** Two-proportion Z-test
- **Outcome:** Conversion rate, Visit rate (same question applied to different metrics)

---

### SECONDARY COMPARISONS (4 tests)
**Questions:** Which email type works better, and does it affect spending?

1. **Mens Email vs Control: Conversion rate**
   - H₀: Conv(Mens) = Conv(Control)
   - H₁: Conv(Mens) ≠ Conv(Control)
   
2. **Womens Email vs Control: Conversion rate**
   - H₀: Conv(Womens) = Conv(Control)
   - H₁: Conv(Womens) ≠ Conv(Control)

3. **Mens Email vs Control: Average Spend**
   - H₀: Spend(Mens) = Spend(Control)
   - H₁: Spend(Mens) ≠ Spend(Control)

4. **Womens Email vs Control: Average Spend**
   - H₀: Spend(Womens) = Spend(Control)
   - H₁: Spend(Womens) ≠ Spend(Control)

**Significance level (Bonferroni corrected):** α = 0.05 / 4 = **0.0125 per test**

---

### EXPLORATORY ANALYSIS (6+ tests)
**Questions:** Does the *alignment* between email and purchase history matter?

**Note:** These are hypothesis-generating. We do NOT apply multiple-testing correction, but findings need confirmation in a dedicated follow-up experiment.

- Match (email aligns with history) vs Control: Conversion & Spend
- Mismatch (email doesn't align) vs Control: Conversion & Spend
- Match vs Mismatch direct comparison: Conversion & Spend
- Mixed/Cross-shoppers vs Control: Conversion & Spend

---

## Why Bonferroni Correction?

With 4 secondary tests and α = 0.05 each, the probability of at least one false positive ≈ 18%.
Bonferroni divides the threshold by 4: 0.05 / 4 = 0.0125.
Now the probability of any false positive ≈ 5% (controlling the family-wise error rate).

For exploratory tests, we relax this (no correction) because we're not making business decisions yet—we're just finding hypotheses.

## Hypothesis Testing: A Quick Refresher

### The Basic Idea
We collect data and ask: "Is the difference we see big enough to be real, or could it just be random noise?"

### Key Concepts

**Null Hypothesis (H₀):** There is no difference; any observed difference is random noise.

**Alternative Hypothesis (H₁):** There is a real difference between groups.

**P-value:** The probability of observing data *at least as extreme* as what we saw, **if H₀ were true**.
- Small p-value → unlikely to see this if there were no difference → reject H₀
- Large p-value → plausible under H₀ → fail to reject H₀

**Significance Level (α):** Our threshold for deciding "small enough to reject H₀"
- Typical: α = 0.05 (5% false positive rate)
- More conservative: α = 0.01 (1% false positive rate)

**Type I Error:** Rejecting H₀ when it's actually true (false positive)
- Rate = α

**Type II Error:** Failing to reject H₀ when it's actually false (false negative)
- Related to statistical power

### Tests We'll Use

**Two-Proportion Z-Test:** Comparing conversion rates (binary outcome) between groups
- Test statistic: Z = (p₁ - p₂) / SE
- 95% CI for difference: (p₁ - p₂) ± 1.96 × SE

**Welch's T-Test:** Comparing average spending between groups (continuous outcome)
- Doesn't assume equal variances
- Robust to non-normal data with large samples
- Test statistic: t = (μ₁ - μ₂) / SE

**Mann-Whitney U Test:** Non-parametric alternative to t-test
- Makes no normality assumption
- Good for robustness checks

### Effect Size
Beyond p-values, we report effect size to show *magnitude* of difference:
- **Cohen's h:** For proportions. h = 2×(arcsin(√p₁) - arcsin(√p₂))
  - Small: 0.2, Medium: 0.5, Large: 0.8
- **Cohen's d:** For means. d = (μ₁ - μ₂) / pooled SD
  - Small: 0.2, Medium: 0.5, Large: 0.8

---

# PART 1: PRIMARY COMPARISON

We test the main business question: **Did email campaigns work overall?**

In [3]:
def two_proportion_ztest(count1, n1, count2, n2):
    """
    Perform a two-proportion Z-test.
    
    Parameters:
    - count1, count2: number of successes in each group
    - n1, n2: sample sizes
    
    Returns:
    - z_stat: Z-statistic
    - p_value: two-tailed p-value
    - ci_diff: 95% CI for (p1 - p2)
    - cohens_h: Cohen's h effect size
    """
    p1 = count1 / n1
    p2 = count2 / n2
    
    # Pooled proportion
    p_pool = (count1 + count2) / (n1 + n2)
    
    # Standard error
    se = np.sqrt(p_pool * (1 - p_pool) * (1/n1 + 1/n2))
    
    # Z-statistic
    z_stat = (p1 - p2) / se if se > 0 else 0
    
    # Two-tailed p-value
    p_value = 2 * (1 - norm.cdf(abs(z_stat)))
    
    # 95% CI for difference
    se_diff = np.sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)
    ci_lower = (p1 - p2) - 1.96 * se_diff
    ci_upper = (p1 - p2) + 1.96 * se_diff
    
    # Cohen's h effect size
    h = 2 * (np.arcsin(np.sqrt(p1)) - np.arcsin(np.sqrt(p2)))
    
    return z_stat, p_value, (ci_lower, ci_upper), p1, p2, h

# PRIMARY: Any Email vs No Email on Conversion
print("="*70)
print("PRIMARY COMPARISON: Any Email vs No Email")
print("Outcome: Conversion Rate")
print("="*70)

# Filter groups
control = df[df['treatment'] == 0]
email = df[df['treatment'] == 1]

print(f"\nSample sizes:")
print(f"  Control (No Email): n = {len(control)}")
print(f"  Email (Any): n = {len(email)}")

# Conversion analysis
control_conv = (control['conversion'] == 1).sum()
email_conv = (email['conversion'] == 1).sum()

control_conv_rate = control_conv / len(control)
email_conv_rate = email_conv / len(email)

z_stat, p_value, ci, p1, p2, h = two_proportion_ztest(
    email_conv, len(email), control_conv, len(control)
)

print(f"\nConversion Rates:")
print(f"  Control: {control_conv_rate:.4f} ({control_conv}/{len(control)})")
print(f"  Email:   {email_conv_rate:.4f} ({email_conv}/{len(email)})")
print(f"  Difference: {email_conv_rate - control_conv_rate:+.4f}")

print(f"\nStatistical Test (Two-Proportion Z-Test):")
print(f"  H₀: Conversion rate (Email) = Conversion rate (Control)")
print(f"  H₁: Conversion rate (Email) ≠ Conversion rate (Control)")
print(f"\n  Z-statistic: {z_stat:.4f}")
print(f"  P-value (two-tailed): {p_value:.6f}")
print(f"  95% CI for difference: [{ci[0]:+.4f}, {ci[1]:+.4f}]")
print(f"  Significance level (α): 0.05")

if p_value < 0.05:
    print(f"\n  ✓ SIGNIFICANT: Reject H₀ (p = {p_value:.6f} < 0.05)")
else:
    print(f"\n  ✗ NOT SIGNIFICANT: Fail to reject H₀ (p = {p_value:.6f} ≥ 0.05)")

# Effect size
if abs(h) < 0.2:
    magnitude = "negligible"
elif abs(h) < 0.5:
    magnitude = "small"
elif abs(h) < 0.8:
    magnitude = "medium"
else:
    magnitude = "large"

print(f"\nEffect Size:")
print(f"  Cohen's h: {h:.4f} ({magnitude})")

# Store for later
z_primary_conv = z_stat
p_primary_conv = p_value
h_primary_conv = h

PRIMARY COMPARISON: Any Email vs No Email
Outcome: Conversion Rate

Sample sizes:
  Control (No Email): n = 21306
  Email (Any): n = 42694

Conversion Rates:
  Control: 0.0057 (122/21306)
  Email:   0.0107 (456/42694)
  Difference: +0.0050

Statistical Test (Two-Proportion Z-Test):
  H₀: Conversion rate (Email) = Conversion rate (Control)
  H₁: Conversion rate (Email) ≠ Conversion rate (Control)

  Z-statistic: 6.2438
  P-value (two-tailed): 0.000000
  95% CI for difference: [+0.0035, +0.0064]
  Significance level (α): 0.05

  ✓ SIGNIFICANT: Reject H₀ (p = 0.000000 < 0.05)

Effect Size:
  Cohen's h: 0.0556 (negligible)


In [4]:
# PRIMARY: Any Email vs No Email on Visit Rate
print("\n" + "="*70)
print("PRIMARY COMPARISON: Any Email vs No Email")
print("Outcome: Visit Rate")
print("="*70)

# Visit analysis
control_visit = (control['visit'] == 1).sum()
email_visit = (email['visit'] == 1).sum()

control_visit_rate = control_visit / len(control)
email_visit_rate = email_visit / len(email)

z_stat_visit, p_value_visit, ci_visit, pv1, pv2, h_visit = two_proportion_ztest(
    email_visit, len(email), control_visit, len(control)
)

print(f"\nVisit Rates:")
print(f"  Control: {control_visit_rate:.4f} ({control_visit}/{len(control)})")
print(f"  Email:   {email_visit_rate:.4f} ({email_visit}/{len(email)})")
print(f"  Difference: {email_visit_rate - control_visit_rate:+.4f}")

print(f"\nStatistical Test (Two-Proportion Z-Test):")
print(f"  H₀: Visit rate (Email) = Visit rate (Control)")
print(f"  H₁: Visit rate (Email) ≠ Visit rate (Control)")
print(f"\n  Z-statistic: {z_stat_visit:.4f}")
print(f"  P-value (two-tailed): {p_value_visit:.6f}")
print(f"  95% CI for difference: [{ci_visit[0]:+.4f}, {ci_visit[1]:+.4f}]")
print(f"  Significance level (α): 0.05")

if p_value_visit < 0.05:
    print(f"\n  ✓ SIGNIFICANT: Reject H₀ (p = {p_value_visit:.6f} < 0.05)")
else:
    print(f"\n  ✗ NOT SIGNIFICANT: Fail to reject H₀ (p = {p_value_visit:.6f} ≥ 0.05)")

if abs(h_visit) < 0.2:
    magnitude = "negligible"
elif abs(h_visit) < 0.5:
    magnitude = "small"
elif abs(h_visit) < 0.8:
    magnitude = "medium"
else:
    magnitude = "large"

print(f"\nEffect Size:")
print(f"  Cohen's h: {h_visit:.4f} ({magnitude})")

# Store for later
z_primary_visit = z_stat_visit
p_primary_visit = p_value_visit
h_primary_visit = h_visit


PRIMARY COMPARISON: Any Email vs No Email
Outcome: Visit Rate

Visit Rates:
  Control: 0.1062 (2262/21306)
  Email:   0.1670 (7132/42694)
  Difference: +0.0609

Statistical Test (Two-Proportion Z-Test):
  H₀: Visit rate (Email) = Visit rate (Control)
  H₁: Visit rate (Email) ≠ Visit rate (Control)

  Z-statistic: 20.5101
  P-value (two-tailed): 0.000000
  95% CI for difference: [+0.0554, +0.0663]
  Significance level (α): 0.05

  ✓ SIGNIFICANT: Reject H₀ (p = 0.000000 < 0.05)

Effect Size:
  Cohen's h: 0.1783 (negligible)


In [5]:
# (Chart cell removed — interactive Plotly version rendered below in the "Blog-Ready Plotly Charts" section.)

---

# PART 2: SECONDARY COMPARISONS

We drill into the primary result: which email type (Mens vs Womens) drove the effect,
and does it affect spending?

**Important:** We apply Bonferroni correction across 4 tests (Mens conv, Womens conv, Mens spend, Womens spend).

**Adjusted significance level:** α = 0.05 / 4 = **0.0125**

This means a test must have p < 0.0125 (not just p < 0.05) to be significant at the secondary level.

In [6]:
print("="*70)
print("SECONDARY COMPARISON: Mens Email vs Control")
print("Outcome: Conversion Rate")
print("="*70)

# Filter segments (exact match: "Mens E-Mail" with no apostrophe)
mens = df[df['segment'] == 'Mens E-Mail']
control = df[df['treatment'] == 0]

print(f"\nSample sizes:")
print(f"  Control: n = {len(control)}")
print(f"  Mens E-Mail: n = {len(mens)}")

# Conversion
mens_conv = (mens['conversion'] == 1).sum()
control_conv = (control['conversion'] == 1).sum()

mens_conv_rate = mens_conv / len(mens)
control_conv_rate = control_conv / len(control)

z_stat_m, p_value_m, ci_m, pm1, pm2, h_m = two_proportion_ztest(
    mens_conv, len(mens), control_conv, len(control)
)

print(f"\nConversion Rates:")
print(f"  Control: {control_conv_rate:.4f} ({control_conv}/{len(control)})")
print(f"  Mens E-Mail: {mens_conv_rate:.4f} ({mens_conv}/{len(mens)})")
print(f"  Difference: {mens_conv_rate - control_conv_rate:+.4f}")

print(f"\nStatistical Test (Two-Proportion Z-Test):")
print(f"  H₀: Conv(Mens) = Conv(Control)")
print(f"  H₁: Conv(Mens) ≠ Conv(Control)")
print(f"\n  Z-statistic: {z_stat_m:.4f}")
print(f"  P-value: {p_value_m:.6f}")
print(f"  95% CI: [{ci_m[0]:+.4f}, {ci_m[1]:+.4f}]")
print(f"  Bonferroni α: 0.0125")

if p_value_m < 0.0125:
    print(f"\n  ✓ SIGNIFICANT: p = {p_value_m:.6f} < 0.0125")
else:
    print(f"\n  ✗ NOT SIGNIFICANT: p = {p_value_m:.6f} ≥ 0.0125")

if abs(h_m) < 0.2:
    mag = "negligible"
elif abs(h_m) < 0.5:
    mag = "small"
elif abs(h_m) < 0.8:
    mag = "medium"
else:
    mag = "large"
print(f"  Cohen's h: {h_m:.4f} ({mag})")

# Store
sec_mens_conv_z = z_stat_m
sec_mens_conv_p = p_value_m
sec_mens_conv_h = h_m

SECONDARY COMPARISON: Mens Email vs Control
Outcome: Conversion Rate

Sample sizes:
  Control: n = 21306
  Mens E-Mail: n = 21307

Conversion Rates:
  Control: 0.0057 (122/21306)
  Mens E-Mail: 0.0125 (267/21307)
  Difference: +0.0068

Statistical Test (Two-Proportion Z-Test):
  H₀: Conv(Mens) = Conv(Control)
  H₁: Conv(Mens) ≠ Conv(Control)

  Z-statistic: 7.3851
  P-value: 0.000000
  95% CI: [+0.0050, +0.0086]
  Bonferroni α: 0.0125

  ✓ SIGNIFICANT: p = 0.000000 < 0.0125
  Cohen's h: 0.0729 (negligible)


In [7]:
print("\n" + "="*70)
print("SECONDARY COMPARISON: Womens Email vs Control")
print("Outcome: Conversion Rate")
print("="*70)

# Filter segments (exact match: "Womens E-Mail")
womens = df[df['segment'] == 'Womens E-Mail']

print(f"\nSample sizes:")
print(f"  Control: n = {len(control)}")
print(f"  Womens E-Mail: n = {len(womens)}")

# Conversion
womens_conv = (womens['conversion'] == 1).sum()
control_conv_total = (control['conversion'] == 1).sum()

womens_conv_rate = womens_conv / len(womens)
control_conv_rate = control_conv_total / len(control)

z_stat_w, p_value_w, ci_w, pw1, pw2, h_w = two_proportion_ztest(
    womens_conv, len(womens), control_conv_total, len(control)
)

print(f"\nConversion Rates:")
print(f"  Control: {control_conv_rate:.4f} ({control_conv_total}/{len(control)})")
print(f"  Womens E-Mail: {womens_conv_rate:.4f} ({womens_conv}/{len(womens)})")
print(f"  Difference: {womens_conv_rate - control_conv_rate:+.4f}")

print(f"\nStatistical Test (Two-Proportion Z-Test):")
print(f"  H₀: Conv(Womens) = Conv(Control)")
print(f"  H₁: Conv(Womens) ≠ Conv(Control)")
print(f"\n  Z-statistic: {z_stat_w:.4f}")
print(f"  P-value: {p_value_w:.6f}")
print(f"  95% CI: [{ci_w[0]:+.4f}, {ci_w[1]:+.4f}]")
print(f"  Bonferroni α: 0.0125")

if p_value_w < 0.0125:
    print(f"\n  ✓ SIGNIFICANT: p = {p_value_w:.6f} < 0.0125")
else:
    print(f"\n  ✗ NOT SIGNIFICANT: p = {p_value_w:.6f} ≥ 0.0125")

if abs(h_w) < 0.2:
    mag = "negligible"
elif abs(h_w) < 0.5:
    mag = "small"
elif abs(h_w) < 0.8:
    mag = "medium"
else:
    mag = "large"
print(f"  Cohen's h: {h_w:.4f} ({mag})")

# Store
sec_womens_conv_z = z_stat_w
sec_womens_conv_p = p_value_w
sec_womens_conv_h = h_w


SECONDARY COMPARISON: Womens Email vs Control
Outcome: Conversion Rate

Sample sizes:
  Control: n = 21306
  Womens E-Mail: n = 21387

Conversion Rates:
  Control: 0.0057 (122/21306)
  Womens E-Mail: 0.0088 (189/21387)
  Difference: +0.0031

Statistical Test (Two-Proportion Z-Test):
  H₀: Conv(Womens) = Conv(Control)
  H₁: Conv(Womens) ≠ Conv(Control)

  Z-statistic: 3.7796
  P-value: 0.000157
  95% CI: [+0.0015, +0.0047]
  Bonferroni α: 0.0125

  ✓ SIGNIFICANT: p = 0.000157 < 0.0125
  Cohen's h: 0.0368 (negligible)


In [8]:
print(f"Saved: {output_dir}/nb02_secondary_conversion.png")

Saved: ../data/outputs/nb02/nb02_secondary_conversion.png


## Spending Analysis: T-Test Assumptions

We now move to spending (a continuous variable), requiring t-tests instead of z-tests.

### When to Use Each Test

**Student's T-Test** (equal variances, rarely correct):
- Assumes both groups have the same variance
- Assumes normal distribution
- Not robust to violations

**Welch's T-Test** (unequal variances):
- Does NOT assume equal variances
- More conservative and safer
- Our default choice

**Mann-Whitney U Test** (non-parametric):
- Makes NO normality assumption
- Uses ranks instead of raw values
- Good robustness check if data is skewed

### Our Strategy

1. Check normality with Shapiro-Wilk test and Q-Q plots
2. Check equal variance with Levene's test
3. Use **Welch's t-test** (the safest bet)
4. Also report **Mann-Whitney U** as a robustness check
5. If both agree, we're confident. If they disagree, be cautious.

In [9]:
print("="*70)
print("ASSUMPTION CHECKS FOR SPENDING ANALYSIS")
print("="*70)

# Remove zero spending for normality check (common in marketing data)
control_spend_nonzero = control[control['spend'] > 0]['spend'].values
email_spend_nonzero = email[email['spend'] > 0]['spend'].values

# Shapiro-Wilk test (null: data is normal)
stat_c, p_c = shapiro(control_spend_nonzero)
stat_e, p_e = shapiro(email_spend_nonzero)

print(f"\nShapiro-Wilk Normality Test (among non-zero spenders):")
print(f"  Control: W = {stat_c:.4f}, p = {p_c:.6f}")
print(f"    → {'Normal' if p_c > 0.05 else 'NOT Normal'} (p {'>' if p_c > 0.05 else '<'} 0.05)")
print(f"  Email:   W = {stat_e:.4f}, p = {p_e:.6f}")
print(f"    → {'Normal' if p_e > 0.05 else 'NOT Normal'} (p {'>' if p_e > 0.05 else '<'} 0.05)")

# Levene's test for equal variance
stat_lev, p_lev = levene(control['spend'], email['spend'])
print(f"\nLevene's Test for Equal Variance:")
print(f"  Statistic: {stat_lev:.4f}, p = {p_lev:.6f}")
print(f"  → Variances are {'EQUAL' if p_lev > 0.05 else 'UNEQUAL'} (p {'>' if p_lev > 0.05 else '<'} 0.05)")

print(f"\n{'='*70}")
print("CONCLUSION:")
print(f"{'='*70}")
print("Data is NOT normally distributed (common in spending data).")
print("We will use Welch's T-Test (robust to non-normality with large n)")
print("+ Mann-Whitney U (non-parametric robustness check).")

# Q-Q plots
print(f"\nQ-Q plots saved: {output_dir}/nb02_qq_plots.png")

ASSUMPTION CHECKS FOR SPENDING ANALYSIS

Shapiro-Wilk Normality Test (among non-zero spenders):
  Control: W = 0.7614, p = 0.000000
    → NOT Normal (p < 0.05)
  Email:   W = 0.7801, p = 0.000000
    → NOT Normal (p < 0.05)

Levene's Test for Equal Variance:
  Statistic: 22.3973, p = 0.000002
  → Variances are UNEQUAL (p < 0.05)

CONCLUSION:
Data is NOT normally distributed (common in spending data).
We will use Welch's T-Test (robust to non-normality with large n)
+ Mann-Whitney U (non-parametric robustness check).

Q-Q plots saved: ../data/outputs/nb02/nb02_qq_plots.png


In [10]:
print("\n" + "="*70)
print("SECONDARY COMPARISON: Mens Email vs Control")
print("Outcome: Average Spending")
print("="*70)

# Welch's t-test
t_stat_m_spend, p_value_m_spend = ttest_ind(mens['spend'], control['spend'], 
                                             equal_var=False)

# Mann-Whitney U test
u_stat_m, p_value_m_mw = mannwhitneyu(mens['spend'], control['spend'], 
                                       alternative='two-sided')

# Effect size (Cohen's d)
def cohens_d(group1, group2):
    n1, n2 = len(group1), len(group2)
    var1, var2 = group1.var(), group2.var()
    pooled_std = np.sqrt(((n1-1)*var1 + (n2-1)*var2) / (n1 + n2 - 2))
    return (group1.mean() - group2.mean()) / pooled_std

d_m_spend = cohens_d(mens['spend'], control['spend'])

print(f"\nSample sizes:")
print(f"  Control: n = {len(control)}")
print(f"  Mens E-Mail: n = {len(mens)}")

print(f"\nDescriptive Statistics:")
print(f"  Control: Mean = ${control['spend'].mean():.2f}, Median = ${control['spend'].median():.2f}, SD = ${control['spend'].std():.2f}")
print(f"  Mens E-Mail: Mean = ${mens['spend'].mean():.2f}, Median = ${mens['spend'].median():.2f}, SD = ${mens['spend'].std():.2f}")
print(f"  Difference in means: ${mens['spend'].mean() - control['spend'].mean():+.2f}")

print(f"\nWelch's T-Test (Primary, more robust):")
print(f"  H₀: Spend(Mens) = Spend(Control)")
print(f"  H₁: Spend(Mens) ≠ Spend(Control)")
print(f"  t-statistic: {t_stat_m_spend:.4f}")
print(f"  P-value: {p_value_m_spend:.6f}")
print(f"  Bonferroni α: 0.0125")

if p_value_m_spend < 0.0125:
    print(f"  ✓ SIGNIFICANT: p = {p_value_m_spend:.6f} < 0.0125")
else:
    print(f"  ✗ NOT SIGNIFICANT: p = {p_value_m_spend:.6f} ≥ 0.0125")

print(f"\nMann-Whitney U Test (Robustness check, non-parametric):")
print(f"  U-statistic: {u_stat_m:.4f}")
print(f"  P-value: {p_value_m_mw:.6f}")
if p_value_m_mw < 0.0125:
    print(f"  ✓ SIGNIFICANT: p = {p_value_m_mw:.6f} < 0.0125")
else:
    print(f"  ✗ NOT SIGNIFICANT: p = {p_value_m_mw:.6f} ≥ 0.0125")

if abs(d_m_spend) < 0.2:
    mag = "negligible"
elif abs(d_m_spend) < 0.5:
    mag = "small"
elif abs(d_m_spend) < 0.8:
    mag = "medium"
else:
    mag = "large"
print(f"\nEffect Size:")
print(f"  Cohen's d: {d_m_spend:.4f} ({mag})")

# Store
sec_mens_spend_t = t_stat_m_spend
sec_mens_spend_p = p_value_m_spend
sec_mens_spend_d = d_m_spend


SECONDARY COMPARISON: Mens Email vs Control
Outcome: Average Spending

Sample sizes:
  Control: n = 21306
  Mens E-Mail: n = 21307

Descriptive Statistics:
  Control: Mean = $0.65, Median = $0.00, SD = $11.59
  Mens E-Mail: Mean = $1.42, Median = $0.00, SD = $17.75
  Difference in means: $+0.77

Welch's T-Test (Primary, more robust):
  H₀: Spend(Mens) = Spend(Control)
  H₁: Spend(Mens) ≠ Spend(Control)
  t-statistic: 5.3001
  P-value: 0.000000
  Bonferroni α: 0.0125
  ✓ SIGNIFICANT: p = 0.000000 < 0.0125

Mann-Whitney U Test (Robustness check, non-parametric):
  U-statistic: 228527081.0000
  P-value: 0.000000
  ✓ SIGNIFICANT: p = 0.000000 < 0.0125

Effect Size:
  Cohen's d: 0.0514 (negligible)


In [11]:
print("\n" + "="*70)
print("SECONDARY COMPARISON: Womens Email vs Control")
print("Outcome: Average Spending")
print("="*70)

# Welch's t-test
t_stat_w_spend, p_value_w_spend = ttest_ind(womens['spend'], control['spend'], 
                                             equal_var=False)

# Mann-Whitney U test
u_stat_w, p_value_w_mw = mannwhitneyu(womens['spend'], control['spend'], 
                                       alternative='two-sided')

# Effect size
d_w_spend = cohens_d(womens['spend'], control['spend'])

print(f"\nSample sizes:")
print(f"  Control: n = {len(control)}")
print(f"  Womens E-Mail: n = {len(womens)}")

print(f"\nDescriptive Statistics:")
print(f"  Control: Mean = ${control['spend'].mean():.2f}, Median = ${control['spend'].median():.2f}, SD = ${control['spend'].std():.2f}")
print(f"  Womens E-Mail: Mean = ${womens['spend'].mean():.2f}, Median = ${womens['spend'].median():.2f}, SD = ${womens['spend'].std():.2f}")
print(f"  Difference in means: ${womens['spend'].mean() - control['spend'].mean():+.2f}")

print(f"\nWelch's T-Test (Primary, more robust):")
print(f"  H₀: Spend(Womens) = Spend(Control)")
print(f"  H₁: Spend(Womens) ≠ Spend(Control)")
print(f"  t-statistic: {t_stat_w_spend:.4f}")
print(f"  P-value: {p_value_w_spend:.6f}")
print(f"  Bonferroni α: 0.0125")

if p_value_w_spend < 0.0125:
    print(f"  ✓ SIGNIFICANT: p = {p_value_w_spend:.6f} < 0.0125")
else:
    print(f"  ✗ NOT SIGNIFICANT: p = {p_value_w_spend:.6f} ≥ 0.0125")

print(f"\nMann-Whitney U Test (Robustness check, non-parametric):")
print(f"  U-statistic: {u_stat_w:.4f}")
print(f"  P-value: {p_value_w_mw:.6f}")
if p_value_w_mw < 0.0125:
    print(f"  ✓ SIGNIFICANT: p = {p_value_w_mw:.6f} < 0.0125")
else:
    print(f"  ✗ NOT SIGNIFICANT: p = {p_value_w_mw:.6f} ≥ 0.0125")

if abs(d_w_spend) < 0.2:
    mag = "negligible"
elif abs(d_w_spend) < 0.5:
    mag = "small"
elif abs(d_w_spend) < 0.8:
    mag = "medium"
else:
    mag = "large"
print(f"\nEffect Size:")
print(f"  Cohen's d: {d_w_spend:.4f} ({mag})")

# Store
sec_womens_spend_t = t_stat_w_spend
sec_womens_spend_p = p_value_w_spend
sec_womens_spend_d = d_w_spend


SECONDARY COMPARISON: Womens Email vs Control
Outcome: Average Spending

Sample sizes:
  Control: n = 21306
  Womens E-Mail: n = 21387

Descriptive Statistics:
  Control: Mean = $0.65, Median = $0.00, SD = $11.59
  Womens E-Mail: Mean = $1.08, Median = $0.00, SD = $15.12
  Difference in means: $+0.42

Welch's T-Test (Primary, more robust):
  H₀: Spend(Womens) = Spend(Control)
  H₁: Spend(Womens) ≠ Spend(Control)
  t-statistic: 3.2564
  P-value: 0.001129
  Bonferroni α: 0.0125
  ✓ SIGNIFICANT: p = 0.001129 < 0.0125

Mann-Whitney U Test (Robustness check, non-parametric):
  U-statistic: 228545030.0000
  P-value: 0.000155
  ✓ SIGNIFICANT: p = 0.000155 < 0.0125

Effect Size:
  Cohen's d: 0.0315 (negligible)


In [12]:
print(f"Saved: {output_dir}/nb02_secondary_spending.png")

Saved: ../data/outputs/nb02/nb02_secondary_spending.png


---

# PART 3: EXPLORATORY ANALYSIS

## Purchase History Match: Do We Send the Right Emails?

**Important Disclaimer:** These analyses are **hypothesis-generating only**.
We do NOT control the false positive rate (no Bonferroni correction).
Any findings must be confirmed in a dedicated follow-up experiment.

### What is Email Match?

The data includes a feature `email_match_simple` based on **purchase department history**, NOT customer gender:

- **Match:** Customer's email aligns with their shopping history
  - Example: A "womens_only" buyer receives "Womens E-Mail"
  
- **Mismatch:** Customer's email doesn't align with their history
  - Example: A "mens_only" buyer receives "Womens E-Mail"

- **Mixed:** Customer is a cross-shopper (buys both departments)

- **Control:** No email sent

### Research Question
Does alignment between email and purchase history improve performance?
This tests whether we're "preaching to the choir" or genuinely reaching the right audience.

In [13]:
print("\n" + "="*70)
print("EXPLORATORY: Match vs Control")
print("(Hypothesis-generating, no multiple-testing correction)")
print("="*70)

# Filter Match and Control
match = df[df['email_match_simple'] == 'Match']
control_exp = df[df['email_match_simple'] == 'Control']

print(f"\nSample sizes:")
print(f"  Control: n = {len(control_exp)}")
print(f"  Match: n = {len(match)}")

# === Conversion ===
match_conv = (match['conversion'] == 1).sum()
control_conv = (control_exp['conversion'] == 1).sum()

match_conv_rate = match_conv / len(match)
control_conv_rate = control_conv / len(control_exp)

z_stat_match_conv, p_value_match_conv, ci_match_conv, _, _, h_match_conv = two_proportion_ztest(
    match_conv, len(match), control_conv, len(control_exp)
)

print(f"\n--- CONVERSION ---")
print(f"Match: {match_conv_rate:.4f} ({match_conv}/{len(match)})")
print(f"Control: {control_conv_rate:.4f} ({control_conv}/{len(control_exp)})")
print(f"Difference: {match_conv_rate - control_conv_rate:+.4f}")
print(f"\nZ-test: z = {z_stat_match_conv:.4f}, p = {p_value_match_conv:.6f}")
if p_value_match_conv < 0.05:
    print(f"✓ p < 0.05 (exploratory)")
else:
    print(f"✗ p ≥ 0.05")
print(f"Cohen's h: {h_match_conv:.4f}")

# === Spending ===
t_stat_match_spend, p_value_match_spend = ttest_ind(match['spend'], control_exp['spend'], 
                                                      equal_var=False)
d_match_spend = cohens_d(match['spend'], control_exp['spend'])

print(f"\n--- SPENDING ---")
print(f"Match Mean: ${match['spend'].mean():.2f}")
print(f"Control Mean: ${control_exp['spend'].mean():.2f}")
print(f"Difference: ${match['spend'].mean() - control_exp['spend'].mean():+.2f}")
print(f"\nWelch's t-test: t = {t_stat_match_spend:.4f}, p = {p_value_match_spend:.6f}")
if p_value_match_spend < 0.05:
    print(f"✓ p < 0.05 (exploratory)")
else:
    print(f"✗ p ≥ 0.05")
print(f"Cohen's d: {d_match_spend:.4f}")

print(f"\n⚠ REMINDER: These are exploratory findings. Need confirmation via follow-up study.")


EXPLORATORY: Match vs Control
(Hypothesis-generating, no multiple-testing correction)

Sample sizes:
  Control: n = 21306
  Match: n = 19205

--- CONVERSION ---
Match: 0.0109 (209/19205)
Control: 0.0057 (122/21306)
Difference: +0.0052

Z-test: z = 5.7568, p = 0.000000
✓ p < 0.05 (exploratory)
Cohen's h: 0.0575

--- SPENDING ---
Match Mean: $1.25
Control Mean: $0.65
Difference: $+0.60

Welch's t-test: t = 4.2420, p = 0.000022
✓ p < 0.05 (exploratory)
Cohen's d: 0.0429

⚠ REMINDER: These are exploratory findings. Need confirmation via follow-up study.


In [14]:
print("\n" + "="*70)
print("EXPLORATORY: Mismatch vs Control")
print("(Hypothesis-generating, no multiple-testing correction)")
print("="*70)

# Filter Mismatch
mismatch = df[df['email_match_simple'] == 'Mismatch']

print(f"\nSample sizes:")
print(f"  Control: n = {len(control_exp)}")
print(f"  Mismatch: n = {len(mismatch)}")

# === Conversion ===
mismatch_conv = (mismatch['conversion'] == 1).sum()
mismatch_conv_rate = mismatch_conv / len(mismatch)

z_stat_mmatch_conv, p_value_mmatch_conv, ci_mmatch_conv, _, _, h_mmatch_conv = two_proportion_ztest(
    mismatch_conv, len(mismatch), control_conv, len(control_exp)
)

print(f"\n--- CONVERSION ---")
print(f"Mismatch: {mismatch_conv_rate:.4f} ({mismatch_conv}/{len(mismatch)})")
print(f"Control: {control_conv_rate:.4f} ({control_conv}/{len(control_exp)})")
print(f"Difference: {mismatch_conv_rate - control_conv_rate:+.4f}")
print(f"\nZ-test: z = {z_stat_mmatch_conv:.4f}, p = {p_value_mmatch_conv:.6f}")
if p_value_mmatch_conv < 0.05:
    print(f"✓ p < 0.05 (exploratory)")
else:
    print(f"✗ p ≥ 0.05")
print(f"Cohen's h: {h_mmatch_conv:.4f}")

# === Spending ===
t_stat_mmatch_spend, p_value_mmatch_spend = ttest_ind(mismatch['spend'], control_exp['spend'], 
                                                        equal_var=False)
d_mmatch_spend = cohens_d(mismatch['spend'], control_exp['spend'])

print(f"\n--- SPENDING ---")
print(f"Mismatch Mean: ${mismatch['spend'].mean():.2f}")
print(f"Control Mean: ${control_exp['spend'].mean():.2f}")
print(f"Difference: ${mismatch['spend'].mean() - control_exp['spend'].mean():+.2f}")
print(f"\nWelch's t-test: t = {t_stat_mmatch_spend:.4f}, p = {p_value_mmatch_spend:.6f}")
if p_value_mmatch_spend < 0.05:
    print(f"✓ p < 0.05 (exploratory)")
else:
    print(f"✗ p ≥ 0.05")
print(f"Cohen's d: {d_mmatch_spend:.4f}")

print(f"\n⚠ REMINDER: These are exploratory findings. Need confirmation via follow-up study.")


EXPLORATORY: Mismatch vs Control
(Hypothesis-generating, no multiple-testing correction)

Sample sizes:
  Control: n = 21306
  Mismatch: n = 19190

--- CONVERSION ---
Mismatch: 0.0085 (163/19190)
Control: 0.0057 (122/21306)
Difference: +0.0028

Z-test: z = 3.3270, p = 0.000878
✓ p < 0.05 (exploratory)
Cohen's h: 0.0331

--- SPENDING ---
Mismatch Mean: $1.04
Control Mean: $0.65
Difference: $+0.39

Welch's t-test: t = 2.8522, p = 0.004344
✓ p < 0.05 (exploratory)
Cohen's d: 0.0288

⚠ REMINDER: These are exploratory findings. Need confirmation via follow-up study.


In [15]:
print("\n" + "="*70)
print("EXPLORATORY: Match vs Mismatch (Direct Comparison)")
print("(Hypothesis-generating, no multiple-testing correction)")
print("="*70)
print("Question: Does email alignment actually improve performance?")

print(f"\nSample sizes:")
print(f"  Match: n = {len(match)}")
print(f"  Mismatch: n = {len(mismatch)}")

# === Conversion ===
match_conv_rate = (match['conversion'] == 1).sum() / len(match)
mismatch_conv_rate = (mismatch['conversion'] == 1).sum() / len(mismatch)

z_stat_mvsmm, p_value_mvsmm, ci_mvsmm, _, _, h_mvsmm = two_proportion_ztest(
    (match['conversion'] == 1).sum(), len(match), 
    (mismatch['conversion'] == 1).sum(), len(mismatch)
)

print(f"\n--- CONVERSION ---")
print(f"Match: {match_conv_rate:.4f}")
print(f"Mismatch: {mismatch_conv_rate:.4f}")
print(f"Difference: {match_conv_rate - mismatch_conv_rate:+.4f}")
print(f"\nZ-test: z = {z_stat_mvsmm:.4f}, p = {p_value_mvsmm:.6f}")
if p_value_mvsmm < 0.05:
    print(f"✓ p < 0.05 (exploratory)")
else:
    print(f"✗ p ≥ 0.05")
print(f"Cohen's h: {h_mvsmm:.4f}")

# === Spending ===
t_stat_mvsmm_spend, p_value_mvsmm_spend = ttest_ind(match['spend'], mismatch['spend'], 
                                                      equal_var=False)
d_mvsmm_spend = cohens_d(match['spend'], mismatch['spend'])

print(f"\n--- SPENDING ---")
print(f"Match Mean: ${match['spend'].mean():.2f}")
print(f"Mismatch Mean: ${mismatch['spend'].mean():.2f}")
print(f"Difference: ${match['spend'].mean() - mismatch['spend'].mean():+.2f}")
print(f"\nWelch's t-test: t = {t_stat_mvsmm_spend:.4f}, p = {p_value_mvsmm_spend:.6f}")
if p_value_mvsmm_spend < 0.05:
    print(f"✓ p < 0.05 (exploratory)")
else:
    print(f"✗ p ≥ 0.05")
print(f"Cohen's d: {d_mvsmm_spend:.4f}")

print(f"\n⚠ REMINDER: These are exploratory findings. Need confirmation via follow-up study.")


EXPLORATORY: Match vs Mismatch (Direct Comparison)
(Hypothesis-generating, no multiple-testing correction)
Question: Does email alignment actually improve performance?

Sample sizes:
  Match: n = 19205
  Mismatch: n = 19190

--- CONVERSION ---
Match: 0.0109
Mismatch: 0.0085
Difference: +0.0024

Z-test: z = 2.3891, p = 0.016892
✓ p < 0.05 (exploratory)
Cohen's h: 0.0244

--- SPENDING ---
Match Mean: $1.25
Mismatch Mean: $1.04
Difference: $+0.21

Welch's t-test: t = 1.3108, p = 0.189939
✗ p ≥ 0.05
Cohen's d: 0.0134

⚠ REMINDER: These are exploratory findings. Need confirmation via follow-up study.


In [16]:
print("\n" + "="*70)
print("EXPLORATORY: Mixed (Cross-shoppers) vs Control")
print("(Hypothesis-generating, no multiple-testing correction)")
print("="*70)
print("Question: How do cross-shoppers respond to email campaigns?")

# Filter Mixed
mixed = df[df['email_match_simple'] == 'Mixed']

print(f"\nSample sizes:")
print(f"  Control: n = {len(control_exp)}")
print(f"  Mixed: n = {len(mixed)}")

# === Conversion ===
mixed_conv = (mixed['conversion'] == 1).sum()
mixed_conv_rate = mixed_conv / len(mixed)

z_stat_mixed, p_value_mixed, ci_mixed, _, _, h_mixed = two_proportion_ztest(
    mixed_conv, len(mixed), control_conv, len(control_exp)
)

print(f"\n--- CONVERSION ---")
print(f"Mixed: {mixed_conv_rate:.4f} ({mixed_conv}/{len(mixed)})")
print(f"Control: {control_conv_rate:.4f} ({control_conv}/{len(control_exp)})")
print(f"Difference: {mixed_conv_rate - control_conv_rate:+.4f}")
print(f"\nZ-test: z = {z_stat_mixed:.4f}, p = {p_value_mixed:.6f}")
if p_value_mixed < 0.05:
    print(f"✓ p < 0.05 (exploratory)")
else:
    print(f"✗ p ≥ 0.05")
print(f"Cohen's h: {h_mixed:.4f}")

# === Spending ===
t_stat_mixed_spend, p_value_mixed_spend = ttest_ind(mixed['spend'], control_exp['spend'], 
                                                      equal_var=False)
d_mixed_spend = cohens_d(mixed['spend'], control_exp['spend'])

print(f"\n--- SPENDING ---")
print(f"Mixed Mean: ${mixed['spend'].mean():.2f}")
print(f"Control Mean: ${control_exp['spend'].mean():.2f}")
print(f"Difference: ${mixed['spend'].mean() - control_exp['spend'].mean():+.2f}")
print(f"\nWelch's t-test: t = {t_stat_mixed_spend:.4f}, p = {p_value_mixed_spend:.6f}")
if p_value_mixed_spend < 0.05:
    print(f"✓ p < 0.05 (exploratory)")
else:
    print(f"✗ p ≥ 0.05")
print(f"Cohen's d: {d_mixed_spend:.4f}")

print(f"\n⚠ REMINDER: These are exploratory findings. Need confirmation via follow-up study.")


EXPLORATORY: Mixed (Cross-shoppers) vs Control
(Hypothesis-generating, no multiple-testing correction)
Question: How do cross-shoppers respond to email campaigns?

Sample sizes:
  Control: n = 21306
  Mixed: n = 4299

--- CONVERSION ---
Mixed: 0.0195 (84/4299)
Control: 0.0057 (122/21306)
Difference: +0.0138

Z-test: z = 9.2481, p = 0.000000
✓ p < 0.05 (exploratory)
Cohen's h: 0.1290

--- SPENDING ---
Mixed Mean: $2.19
Control Mean: $0.65
Difference: $+1.53

Welch's t-test: t = 4.3645, p = 0.000013
✓ p < 0.05 (exploratory)
Cohen's d: 0.1094

⚠ REMINDER: These are exploratory findings. Need confirmation via follow-up study.


In [17]:
# (Chart cell removed — interactive Plotly version rendered below in the "Blog-Ready Plotly Charts" section.)

## Effect Size Interpretation

Beyond p-values, effect size tells us the **magnitude** of the difference.
A tiny effect might be statistically significant (p < 0.05) in a large sample,
but practically meaningless.

### Cohen's h (for proportions/conversions)
Measures difference between two proportions.

- **h < 0.2:** Negligible effect
- **0.2 ≤ h < 0.5:** Small effect
- **0.5 ≤ h < 0.8:** Medium effect
- **h ≥ 0.8:** Large effect

### Cohen's d (for means/spending)
Measures difference between two group means, standardized by pooled SD.

- **d < 0.2:** Negligible effect
- **0.2 ≤ d < 0.5:** Small effect
- **0.5 ≤ d < 0.8:** Medium effect
- **d ≥ 0.8:** Large effect

### Combined Interpretation
For strong evidence, we want:
1. **Statistically significant** (p-value below threshold)
2. **Practically meaningful** effect size (d or h > 0.2)
3. **Consistent direction** (same sign across related tests)
4. **Robust** (confirmed by multiple tests, e.g., t-test + Mann-Whitney)

In [18]:
# Build comprehensive results table
results = []

# PRIMARY
results.append({
    'Level': 'PRIMARY',
    'Outcome': 'Conversion',
    'Comparison': 'Any Email vs Control',
    'Control_Rate': f'{control_conv_rate:.4f}',
    'Treatment_Rate': f'{email_conv_rate:.4f}',
    'Difference': f'{email_conv_rate - control_conv_rate:+.4f}',
    'Test_Statistic': f'{z_primary_conv:.4f}',
    'P_Value': f'{p_primary_conv:.6f}',
    'Alpha_Threshold': '0.0500',
    'Significant': 'Yes' if p_primary_conv < 0.05 else 'No',
    'Effect_Size': f'{h_primary_conv:.4f}',
    'Effect_Magnitude': 'Small' if abs(h_primary_conv) < 0.5 else ('Medium' if abs(h_primary_conv) < 0.8 else 'Large')
})

results.append({
    'Level': 'PRIMARY',
    'Outcome': 'Visit',
    'Comparison': 'Any Email vs Control',
    'Control_Rate': f'{control_visit_rate:.4f}',
    'Treatment_Rate': f'{email_visit_rate:.4f}',
    'Difference': f'{email_visit_rate - control_visit_rate:+.4f}',
    'Test_Statistic': f'{z_primary_visit:.4f}',
    'P_Value': f'{p_primary_visit:.6f}',
    'Alpha_Threshold': '0.0500',
    'Significant': 'Yes' if p_primary_visit < 0.05 else 'No',
    'Effect_Size': f'{h_primary_visit:.4f}',
    'Effect_Magnitude': 'Small' if abs(h_primary_visit) < 0.5 else ('Medium' if abs(h_primary_visit) < 0.8 else 'Large')
})

# SECONDARY
results.append({
    'Level': 'SECONDARY',
    'Outcome': 'Conversion',
    'Comparison': 'Mens Email vs Control',
    'Control_Rate': f'{control_conv_rate:.4f}',
    'Treatment_Rate': f'{mens_conv_rate:.4f}',
    'Difference': f'{mens_conv_rate - control_conv_rate:+.4f}',
    'Test_Statistic': f'{sec_mens_conv_z:.4f}',
    'P_Value': f'{sec_mens_conv_p:.6f}',
    'Alpha_Threshold': '0.0125',
    'Significant': 'Yes' if sec_mens_conv_p < 0.0125 else 'No',
    'Effect_Size': f'{sec_mens_conv_h:.4f}',
    'Effect_Magnitude': 'Small' if abs(sec_mens_conv_h) < 0.5 else ('Medium' if abs(sec_mens_conv_h) < 0.8 else 'Large')
})

results.append({
    'Level': 'SECONDARY',
    'Outcome': 'Conversion',
    'Comparison': 'Womens Email vs Control',
    'Control_Rate': f'{control_conv_rate:.4f}',
    'Treatment_Rate': f'{womens_conv_rate:.4f}',
    'Difference': f'{womens_conv_rate - control_conv_rate:+.4f}',
    'Test_Statistic': f'{sec_womens_conv_z:.4f}',
    'P_Value': f'{sec_womens_conv_p:.6f}',
    'Alpha_Threshold': '0.0125',
    'Significant': 'Yes' if sec_womens_conv_p < 0.0125 else 'No',
    'Effect_Size': f'{sec_womens_conv_h:.4f}',
    'Effect_Magnitude': 'Small' if abs(sec_womens_conv_h) < 0.5 else ('Medium' if abs(sec_womens_conv_h) < 0.8 else 'Large')
})

results.append({
    'Level': 'SECONDARY',
    'Outcome': 'Spending',
    'Comparison': 'Mens Email vs Control',
    'Control_Rate': f'${control["spend"].mean():.2f}',
    'Treatment_Rate': f'${mens["spend"].mean():.2f}',
    'Difference': f'${mens["spend"].mean() - control["spend"].mean():+.2f}',
    'Test_Statistic': f'{sec_mens_spend_t:.4f}',
    'P_Value': f'{sec_mens_spend_p:.6f}',
    'Alpha_Threshold': '0.0125',
    'Significant': 'Yes' if sec_mens_spend_p < 0.0125 else 'No',
    'Effect_Size': f'{sec_mens_spend_d:.4f}',
    'Effect_Magnitude': 'Small' if abs(sec_mens_spend_d) < 0.5 else ('Medium' if abs(sec_mens_spend_d) < 0.8 else 'Large')
})

results.append({
    'Level': 'SECONDARY',
    'Outcome': 'Spending',
    'Comparison': 'Womens Email vs Control',
    'Control_Rate': f'${control["spend"].mean():.2f}',
    'Treatment_Rate': f'${womens["spend"].mean():.2f}',
    'Difference': f'${womens["spend"].mean() - control["spend"].mean():+.2f}',
    'Test_Statistic': f'{sec_womens_spend_t:.4f}',
    'P_Value': f'{sec_womens_spend_p:.6f}',
    'Alpha_Threshold': '0.0125',
    'Significant': 'Yes' if sec_womens_spend_p < 0.0125 else 'No',
    'Effect_Size': f'{sec_womens_spend_d:.4f}',
    'Effect_Magnitude': 'Small' if abs(sec_womens_spend_d) < 0.5 else ('Medium' if abs(sec_womens_spend_d) < 0.8 else 'Large')
})

# EXPLORATORY
results.append({
    'Level': 'EXPLORATORY',
    'Outcome': 'Conversion',
    'Comparison': 'Match vs Control',
    'Control_Rate': f'{control_conv_rate:.4f}',
    'Treatment_Rate': f'{match_conv_rate:.4f}',
    'Difference': f'{match_conv_rate - control_conv_rate:+.4f}',
    'Test_Statistic': f'{z_stat_match_conv:.4f}',
    'P_Value': f'{p_value_match_conv:.6f}',
    'Alpha_Threshold': 'None',
    'Significant': 'Yes' if p_value_match_conv < 0.05 else 'No',
    'Effect_Size': f'{h_match_conv:.4f}',
    'Effect_Magnitude': 'Small' if abs(h_match_conv) < 0.5 else ('Medium' if abs(h_match_conv) < 0.8 else 'Large')
})

results.append({
    'Level': 'EXPLORATORY',
    'Outcome': 'Spending',
    'Comparison': 'Match vs Control',
    'Control_Rate': f'${control_exp["spend"].mean():.2f}',
    'Treatment_Rate': f'${match["spend"].mean():.2f}',
    'Difference': f'${match["spend"].mean() - control_exp["spend"].mean():+.2f}',
    'Test_Statistic': f'{t_stat_match_spend:.4f}',
    'P_Value': f'{p_value_match_spend:.6f}',
    'Alpha_Threshold': 'None',
    'Significant': 'Yes' if p_value_match_spend < 0.05 else 'No',
    'Effect_Size': f'{d_match_spend:.4f}',
    'Effect_Magnitude': 'Small' if abs(d_match_spend) < 0.5 else ('Medium' if abs(d_match_spend) < 0.8 else 'Large')
})

results.append({
    'Level': 'EXPLORATORY',
    'Outcome': 'Conversion',
    'Comparison': 'Mismatch vs Control',
    'Control_Rate': f'{control_conv_rate:.4f}',
    'Treatment_Rate': f'{mismatch_conv_rate:.4f}',
    'Difference': f'{mismatch_conv_rate - control_conv_rate:+.4f}',
    'Test_Statistic': f'{z_stat_mmatch_conv:.4f}',
    'P_Value': f'{p_value_mmatch_conv:.6f}',
    'Alpha_Threshold': 'None',
    'Significant': 'Yes' if p_value_mmatch_conv < 0.05 else 'No',
    'Effect_Size': f'{h_mmatch_conv:.4f}',
    'Effect_Magnitude': 'Small' if abs(h_mmatch_conv) < 0.5 else ('Medium' if abs(h_mmatch_conv) < 0.8 else 'Large')
})

results.append({
    'Level': 'EXPLORATORY',
    'Outcome': 'Spending',
    'Comparison': 'Mismatch vs Control',
    'Control_Rate': f'${control_exp["spend"].mean():.2f}',
    'Treatment_Rate': f'${mismatch["spend"].mean():.2f}',
    'Difference': f'${mismatch["spend"].mean() - control_exp["spend"].mean():+.2f}',
    'Test_Statistic': f'{t_stat_mmatch_spend:.4f}',
    'P_Value': f'{p_value_mmatch_spend:.6f}',
    'Alpha_Threshold': 'None',
    'Significant': 'Yes' if p_value_mmatch_spend < 0.05 else 'No',
    'Effect_Size': f'{d_mmatch_spend:.4f}',
    'Effect_Magnitude': 'Small' if abs(d_mmatch_spend) < 0.5 else ('Medium' if abs(d_mmatch_spend) < 0.8 else 'Large')
})

results.append({
    'Level': 'EXPLORATORY',
    'Outcome': 'Conversion',
    'Comparison': 'Match vs Mismatch',
    'Control_Rate': f'{mismatch_conv_rate:.4f}',
    'Treatment_Rate': f'{match_conv_rate:.4f}',
    'Difference': f'{match_conv_rate - mismatch_conv_rate:+.4f}',
    'Test_Statistic': f'{z_stat_mvsmm:.4f}',
    'P_Value': f'{p_value_mvsmm:.6f}',
    'Alpha_Threshold': 'None',
    'Significant': 'Yes' if p_value_mvsmm < 0.05 else 'No',
    'Effect_Size': f'{h_mvsmm:.4f}',
    'Effect_Magnitude': 'Small' if abs(h_mvsmm) < 0.5 else ('Medium' if abs(h_mvsmm) < 0.8 else 'Large')
})

results.append({
    'Level': 'EXPLORATORY',
    'Outcome': 'Spending',
    'Comparison': 'Match vs Mismatch',
    'Control_Rate': f'${mismatch["spend"].mean():.2f}',
    'Treatment_Rate': f'${match["spend"].mean():.2f}',
    'Difference': f'${match["spend"].mean() - mismatch["spend"].mean():+.2f}',
    'Test_Statistic': f'{t_stat_mvsmm_spend:.4f}',
    'P_Value': f'{p_value_mvsmm_spend:.6f}',
    'Alpha_Threshold': 'None',
    'Significant': 'Yes' if p_value_mvsmm_spend < 0.05 else 'No',
    'Effect_Size': f'{d_mvsmm_spend:.4f}',
    'Effect_Magnitude': 'Small' if abs(d_mvsmm_spend) < 0.5 else ('Medium' if abs(d_mvsmm_spend) < 0.8 else 'Large')
})

results.append({
    'Level': 'EXPLORATORY',
    'Outcome': 'Conversion',
    'Comparison': 'Mixed vs Control',
    'Control_Rate': f'{control_conv_rate:.4f}',
    'Treatment_Rate': f'{mixed_conv_rate:.4f}',
    'Difference': f'{mixed_conv_rate - control_conv_rate:+.4f}',
    'Test_Statistic': f'{z_stat_mixed:.4f}',
    'P_Value': f'{p_value_mixed:.6f}',
    'Alpha_Threshold': 'None',
    'Significant': 'Yes' if p_value_mixed < 0.05 else 'No',
    'Effect_Size': f'{h_mixed:.4f}',
    'Effect_Magnitude': 'Small' if abs(h_mixed) < 0.5 else ('Medium' if abs(h_mixed) < 0.8 else 'Large')
})

results.append({
    'Level': 'EXPLORATORY',
    'Outcome': 'Spending',
    'Comparison': 'Mixed vs Control',
    'Control_Rate': f'${control_exp["spend"].mean():.2f}',
    'Treatment_Rate': f'${mixed["spend"].mean():.2f}',
    'Difference': f'${mixed["spend"].mean() - control_exp["spend"].mean():+.2f}',
    'Test_Statistic': f'{t_stat_mixed_spend:.4f}',
    'P_Value': f'{p_value_mixed_spend:.6f}',
    'Alpha_Threshold': 'None',
    'Significant': 'Yes' if p_value_mixed_spend < 0.05 else 'No',
    'Effect_Size': f'{d_mixed_spend:.4f}',
    'Effect_Magnitude': 'Small' if abs(d_mixed_spend) < 0.5 else ('Medium' if abs(d_mixed_spend) < 0.8 else 'Large')
})

results_df = pd.DataFrame(results)
results_df.to_csv(f'{output_dir}/nb02_frequentist_results.csv', index=False)

print("\n" + "="*100)
print("COMPREHENSIVE RESULTS SUMMARY")
print("="*100)
print(results_df.to_string(index=False))
print("\n" + "="*100)
print(f"Saved to: {output_dir}/nb02_frequentist_results.csv")



COMPREHENSIVE RESULTS SUMMARY
      Level    Outcome              Comparison Control_Rate Treatment_Rate Difference Test_Statistic  P_Value Alpha_Threshold Significant Effect_Size Effect_Magnitude
    PRIMARY Conversion    Any Email vs Control       0.0057         0.0107    +0.0050         6.2438 0.000000          0.0500         Yes      0.0556            Small
    PRIMARY      Visit    Any Email vs Control       0.1062         0.1670    +0.0609        20.5101 0.000000          0.0500         Yes      0.1783            Small
  SECONDARY Conversion   Mens Email vs Control       0.0057         0.0125    +0.0068         7.3851 0.000000          0.0125         Yes      0.0729            Small
  SECONDARY Conversion Womens Email vs Control       0.0057         0.0088    +0.0031         3.7796 0.000157          0.0125         Yes      0.0368            Small
  SECONDARY   Spending   Mens Email vs Control        $0.65          $1.42     $+0.77         5.3001 0.000000          0.0125         

In [19]:
# (Chart cell removed — interactive Plotly version rendered below in the "Blog-Ready Plotly Charts" section.)

---

# Key Takeaways

## PRIMARY LEVEL: Did Email Campaigns Work?

Review the p-values and effect sizes for the primary tests (Any Email vs Control).
- If **p < 0.05 AND effect size is non-negligible**, email campaigns had a real impact overall.
- If **p ≥ 0.05**, we cannot claim the campaigns worked (lack of evidence).
- Effect size matters: even a significant p-value is weak if Cohen's h or d < 0.2.

## SECONDARY LEVEL: Which Email Type Performed Better?

Look at Mens Email vs Control and Womens Email vs Control (both conversion and spending).
- Remember: these tests use **α = 0.0125** (stricter than primary).
- Only results with **p < 0.0125** are statistically significant at the secondary level.
- Bonferroni correction prevents false claims about which email variant "works best."

## EXPLORATORY LEVEL: Does Email Alignment Matter?

The Match vs Mismatch analyses explore whether targeting based on purchase history helps.
- **These are ideas for future experiments, not confirmed findings.**
- No correction applied (no Bonferroni), so p < 0.05 is suggestive but not conclusive.
- Any interesting patterns should be tested in a dedicated follow-up experiment.

---

## What to Report to Stakeholders

**Frame the analysis clearly:**
- "We had one primary question, four pre-planned secondary comparisons, and exploratory sub-analyses."
- "Our primary comparison tests whether ANY email helped. Secondary tests drill into WHICH email type."
- "The exploratory analyses generate hypotheses (email alignment, cross-shopper behavior) for future testing."

**If primary result is significant:**
- "Email campaigns improved [conversion/visits/spending] (p < 0.05, effect size = X)."
- "This effect appears driven by [Mens/Womens] email (if secondary is also significant)."
- "However, [other observations from exploratory] suggest additional opportunities worth testing."

**If primary result is NOT significant:**
- "Email campaigns did not produce a detectable effect overall (p ≥ 0.05)."
- "Secondary analyses show [details], but these are exploratory given the null primary result."
- "Implications: Either the email strategy needs redesign, or effect is too small to detect with current sample size."

---

## Next Steps

1. **Validate primary finding:** If significant, replicate in a new holdout sample or test period.
2. **Test exploratory hypotheses:** Design a dedicated experiment around email alignment or other promising patterns.
3. **Dig into mechanisms:** Why did email (or not) work? Analyze open rates, click rates, timing, content.
4. **Power analysis:** If effect size is small, consider whether larger sample is needed to detect it reliably.
5. **Subgroup analysis:** Conduct planned (not post-hoc) analysis by customer segment, RFM tier, etc.

---

## Common Pitfalls to Avoid

- ✗ **P-hacking:** Running 50 tests and reporting only the significant ones. (We avoided this with our pre-specified plan.)
- ✗ **Ignoring effect size:** A p-value of 0.04 with h = 0.05 is not practically meaningful.
- ✗ **Overgeneralizing exploratory findings:** Just because Match vs Control is significant doesn't mean we should send only matched emails—test it first!
- ✗ **Forgetting multiple testing correction:** Secondary tests need stricter thresholds.
- ✗ **Confusing absence of evidence with evidence of absence:** p ≥ 0.05 means "no strong evidence," not "proved null hypothesis."

---

## Statistical Recap

| Level | Tests | Significance Level | Correction | Purpose |
|-------|-------|-------------------|-----------|---------|
| **Primary** | 1-2 | α = 0.05 | None | Main business question |
| **Secondary** | 4 | α = 0.0125 | Bonferroni | Pre-planned follow-ups |
| **Exploratory** | 6+ | None (p < 0.05 suggestive) | None | Hypothesis generation |

Use this framework to maintain scientific rigor while staying responsive to business needs.

---

## Blog-Ready Plotly Charts

The cells below regenerate the charts from this notebook as responsive Plotly
HTML files for embedding in the blog post. They are **self-contained**: each
one re-loads the clean dataset from nb01 and re-derives the statistics it
needs, so you can run this section in isolation.

Outputs are written to `data/outputs/nb##/` with the suffix `_interactive.html`.

**Required packages:** `plotly` (install with `pip install plotly` if missing).

In [20]:
# ============================================================
# Blog-Ready Plotly Charts — self-contained, embed-friendly
# ============================================================
# These cells produce responsive Plotly HTML files for the blog post.
# They re-load from the nb01 clean CSV and re-derive stats so the section
# runs standalone. Each figure uses:
#   - include_plotlyjs='cdn' (single shared CDN load on the blog page)
#   - config={'responsive': True} so it resizes to container width
#   - automargin=True on axes + generous margins so labels never clip
#   - rotated tick labels on long categories, headroom for outside labels
import os, numpy as np, pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "notebook_connected"  # inline-render Plotly in cell output

OUT_DIR = os.path.abspath("../data/outputs/nb02")
os.makedirs(OUT_DIR, exist_ok=True)
CLEAN_CSV = os.path.abspath("../data/outputs/nb01/nb01_hillstrom_clean.csv")
df_blog = pd.read_csv(CLEAN_CSV)

# Shared palette aligned with nb01 Plotly charts
COLORS = {
    "Mens E-Mail": "#4C8BB8", "Womens E-Mail": "#5FA85F", "No E-Mail": "#E89B4C",
    "Match": "#2ECC71", "Mismatch": "#E74C3C", "Mixed": "#F39C12", "Control": "#95A5A6",
    "Treatment (Any Email)": "#4C8BB8",
}
PLOTLY_KW = dict(include_plotlyjs="cdn", full_html=True,
                 config={"responsive": True, "displaylogo": False})
BASE_LAYOUT = dict(template="plotly_white",
                   font=dict(family="Arial, sans-serif", size=13),
                   title_x=0.5,
                   margin=dict(l=70, r=40, t=90, b=90),
                   hoverlabel=dict(bgcolor="white", font_size=12))
print(f"Blog-ready Plotly charts will be written to: {OUT_DIR}")


Blog-ready Plotly charts will be written to: /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/ab_testing/data/outputs/nb02


In [21]:
# Chart 1: Visit + Conversion rates with 95% CIs — forest plot
from scipy import stats as spstats

control = df_blog[df_blog["segment"] == "No E-Mail"]
rows = []
for metric in ["visit", "conversion"]:
    p_c = control[metric].mean(); n_c = len(control)
    for seg in ["Mens E-Mail", "Womens E-Mail"]:
        g = df_blog[df_blog["segment"] == seg]
        p_t = g[metric].mean(); n_t = len(g)
        diff = p_t - p_c
        se = np.sqrt(p_t*(1-p_t)/n_t + p_c*(1-p_c)/n_c)
        lo, hi = diff - 1.96*se, diff + 1.96*se
        z = diff / se if se > 0 else 0
        p = 2 * (1 - spstats.norm.cdf(abs(z)))
        rows.append(dict(metric=metric, arm=seg, diff=diff*100, lo=lo*100, hi=hi*100, p=p))
eff = pd.DataFrame(rows)

labels = [f"{r.arm} — {r.metric}" for r in eff.itertuples()]
colors = ["#4C8BB8" if r.metric == "visit" else "#5FA85F" for r in eff.itertuples()]
y_pos = list(range(len(eff)))

fig = go.Figure()
for i, r in enumerate(eff.itertuples()):
    fig.add_trace(go.Scatter(
        x=[r.lo, r.hi], y=[i, i], mode="lines",
        line=dict(color=colors[i], width=4),
        hoverinfo="skip", showlegend=False))
    fig.add_trace(go.Scatter(
        x=[r.diff], y=[i], mode="markers+text",
        marker=dict(size=16, color=colors[i],
                    line=dict(color="black", width=1.5), symbol="diamond"),
        text=[f"  {r.diff:+.2f} pp  (p={r.p:.2e})"],
        textposition="middle right",
        textfont=dict(size=12),
        hovertemplate=(f"<b>{r.arm} ({r.metric})</b><br>"
                       "Lift: %{x:.2f} pp<br>"
                       f"95% CI: [{r.lo:.2f}, {r.hi:.2f}]<br>"
                       f"p = {r.p:.2e}<extra></extra>"),
        showlegend=False))

fig.add_vline(x=0, line_dash="dash", line_color="#555", annotation_text="No effect",
              annotation_position="top")
fig.update_yaxes(tickvals=y_pos, ticktext=labels, automargin=True,
                 autorange="reversed")
fig.update_xaxes(title="Lift vs Control (percentage points)",
                 automargin=True, zeroline=False,
                 range=[min(eff["lo"])-1.5, max(eff["hi"])+3.5])
fig.update_layout(**{**BASE_LAYOUT, "margin": dict(l=200, r=60, t=90, b=80)},
                  title="Effect Sizes with 95% Confidence Intervals — Forest Plot",
                  height=520, showlegend=False)
fig.write_html(os.path.join(OUT_DIR, "nb02_forest_plot_interactive.html"), **PLOTLY_KW)
fig.show()
print("  ✓ nb02_forest_plot_interactive.html")


  ✓ nb02_forest_plot_interactive.html


In [22]:
from scipy import stats as spstats
# Chart 2: Power curve — detectable effect size vs sample size
# Assumes baseline conversion of control; shows minimum detectable lift at power=0.8
baseline = df_blog[df_blog["segment"] == "No E-Mail"]["conversion"].mean()
alpha = 0.05
power_target = 0.80
z_a = spstats.norm.ppf(1 - alpha/2)
z_b = spstats.norm.ppf(power_target)

sample_sizes = np.linspace(500, 25000, 80)
def mde(n, p0, z_a, z_b):
    # Approximate MDE for two-proportion test (equal n per arm)
    # n per arm; returns min detectable absolute lift
    sigma = np.sqrt(2 * p0 * (1 - p0) / n)
    return (z_a + z_b) * sigma

mdes = [mde(n, baseline, z_a, z_b) * 100 for n in sample_sizes]
observed_n = len(df_blog[df_blog["segment"] == "Mens E-Mail"])
observed_mde = mde(observed_n, baseline, z_a, z_b) * 100

fig = go.Figure()
fig.add_trace(go.Scatter(x=sample_sizes, y=mdes, mode="lines",
                         line=dict(color="#4C8BB8", width=3),
                         name="Min detectable lift (80% power)",
                         hovertemplate="n per arm: %{x:,.0f}<br>MDE: %{y:.3f} pp<extra></extra>"))
fig.add_vline(x=observed_n, line_dash="dash", line_color="red",
              annotation_text=f"Hillstrom n per arm = {observed_n:,}",
              annotation_position="top right")
fig.add_hline(y=observed_mde, line_dash="dot", line_color="red")
fig.update_layout(**BASE_LAYOUT,
                  title=f"Power Curve — Minimum Detectable Lift at 80% Power (baseline = {baseline*100:.2f}%)",
                  xaxis=dict(title="Sample Size Per Arm", automargin=True),
                  yaxis=dict(title="Min Detectable Absolute Lift (percentage points)",
                             automargin=True), height=500)
fig.write_html(os.path.join(OUT_DIR, "nb02_power_curve_interactive.html"), **PLOTLY_KW)
fig.show()
print("  ✓ nb02_power_curve_interactive.html")


  ✓ nb02_power_curve_interactive.html


In [23]:
from scipy import stats as spstats
# Chart 3: Q-Q plot of spend (among spenders) by segment — diagnostic for t-test normality
fig = go.Figure()
for seg in ["Mens E-Mail", "Womens E-Mail", "No E-Mail"]:
    data = df_blog[(df_blog["segment"] == seg) & (df_blog["spend"] > 0)]["spend"]
    if len(data) < 5:
        continue
    theo_q, emp_q = spstats.probplot(data, dist="norm", fit=False)
    fig.add_trace(go.Scatter(x=theo_q, y=emp_q, mode="markers",
                             marker=dict(color=COLORS[seg], size=5, opacity=0.6),
                             name=seg,
                             hovertemplate=f"<b>{seg}</b><br>Theo q: %{{x:.2f}}<br>Sample q: $%{{y:.2f}}<extra></extra>"))
# Reference line
all_sp = df_blog[df_blog["spend"] > 0]["spend"]
ref = np.linspace(all_sp.min(), all_sp.max(), 10)
fig.add_trace(go.Scatter(x=spstats.norm.ppf(np.linspace(0.01, 0.99, 10)),
                         y=np.quantile(all_sp, np.linspace(0.01, 0.99, 10)),
                         mode="lines", line=dict(color="black", dash="dash"),
                         name="Normal ref", hoverinfo="skip"))
fig.update_layout(**BASE_LAYOUT,
                  title="Q-Q Plot of Spend (Among Spenders) — Normality Diagnostic",
                  xaxis=dict(title="Theoretical Normal Quantile", automargin=True),
                  yaxis=dict(title="Sample Spend Quantile ($)", automargin=True),
                  height=520,
                  legend=dict(orientation="h", y=-0.2, x=0.5, xanchor="center"))
fig.write_html(os.path.join(OUT_DIR, "nb02_qq_plot_interactive.html"), **PLOTLY_KW)
fig.show()
print("  ✓ nb02_qq_plot_interactive.html")


  ✓ nb02_qq_plot_interactive.html


### Results-Display Charts (embed-ready summaries of the numerical outputs)

The three cells below turn the printed test statistics into blog-ready Plotly
visualizations: a formatted results **table**, a **p-value evidence chart**,
and an **effect-size chart** with interpretation bands.

In [24]:
# Results Display 1: Per-comparison detail tables (one "card" per test)
# + a comprehensive summary table including Exploratory rows.
from scipy.stats import norm, ttest_ind

def ztest(n1, x1, n2, x2):
    p1, p2 = x1/n1, x2/n2
    pool = (x1+x2)/(n1+n2)
    se = np.sqrt(pool*(1-pool)*(1/n1 + 1/n2))
    z = (p1-p2)/se if se>0 else 0
    p = 2*(1-norm.cdf(abs(z)))
    se_diff = np.sqrt(p1*(1-p1)/n1 + p2*(1-p2)/n2)
    ci = (p1-p2 - 1.96*se_diff, p1-p2 + 1.96*se_diff)
    h = 2*(np.arcsin(np.sqrt(p1)) - np.arcsin(np.sqrt(p2)))
    return dict(p1=p1, p2=p2, x1=x1, x2=x2, n1=n1, n2=n2,
                diff=p1-p2, z=z, p=p, ci=ci, h=h)

def welch(a, b):
    t_stat, p = ttest_ind(a, b, equal_var=False)
    diff = a.mean() - b.mean()
    pooled = np.sqrt((a.var(ddof=1) + b.var(ddof=1))/2)
    d = diff / pooled if pooled>0 else 0
    return dict(diff=diff, t=t_stat, p=p, d=d, m1=a.mean(), m2=b.mean(), n1=len(a), n2=len(b))

def eff_label(h):
    a = abs(h)
    if a < 0.2: return "negligible"
    if a < 0.5: return "small"
    if a < 0.8: return "medium"
    return "large"

def detail_card(title, subtitle, pairs, conclusion, alpha):
    """Render a two-column go.Table card matching the text-block format."""
    labels = [p[0] for p in pairs]
    values = [p[1] for p in pairs]
    # alternate row fill
    row_fill = ["#F8F9F9" if i%2==0 else "white" for i in range(len(labels))]
    fig = go.Figure(data=[go.Table(
        columnwidth=[260, 520],
        header=dict(values=[f"<b>{title}</b>", f"<b>{subtitle}</b>"],
                    fill_color="#2C3E50",
                    font=dict(color="white", size=13),
                    align="left", height=36),
        cells=dict(values=[labels, values],
                   fill_color=[row_fill, row_fill],
                   align=["left","left"], font=dict(size=12, family="monospace"),
                   height=28))])
    fig.update_layout(**{**BASE_LAYOUT, "margin": dict(l=20, r=20, t=40, b=20)},
                      height=36 + 28*len(labels) + 60,
                      title=None)
    return fig

# Pre-computed groups
any_email = df_blog[df_blog["segment"] != "No E-Mail"]
control   = df_blog[df_blog["segment"] == "No E-Mail"]
mens_g    = df_blog[df_blog["segment"] == "Mens E-Mail"]
womens_g  = df_blog[df_blog["segment"] == "Womens E-Mail"]

# ------- Primary cards -------
r_visit = ztest(len(any_email), any_email["visit"].sum(),
                len(control),  control["visit"].sum())
r_conv  = ztest(len(any_email), any_email["conversion"].sum(),
                len(control),  control["conversion"].sum())
r_spend = welch(any_email["spend"], control["spend"])

primary_visit = detail_card(
    "PRIMARY COMPARISON", "Any Email vs No Email — Visit Rate",
    [("Sample sizes",            f"Control n={r_visit['n2']:,} | Email n={r_visit['n1']:,}"),
     ("Visit rate — Control",    f"{r_visit['p2']:.4f}  ({r_visit['x2']:,}/{r_visit['n2']:,})"),
     ("Visit rate — Email",      f"{r_visit['p1']:.4f}  ({r_visit['x1']:,}/{r_visit['n1']:,})"),
     ("Difference",              f"{r_visit['diff']:+.4f}"),
     ("Test",                    "Two-Proportion Z-Test (two-tailed)"),
     ("Hypotheses",              "H₀: p_email = p_ctrl   H₁: p_email ≠ p_ctrl"),
     ("Z-statistic",             f"{r_visit['z']:.4f}"),
     ("P-value",                 f"{r_visit['p']:.2e}"),
     ("95% CI for difference",   f"[{r_visit['ci'][0]:+.4f}, {r_visit['ci'][1]:+.4f}]"),
     ("Alpha (α)",               "0.05"),
     ("Conclusion",               ("✓ SIGNIFICANT — reject H₀" if r_visit["p"]<0.05 else "✗ not significant")),
     ("Effect size (Cohen's h)", f"{r_visit['h']:+.4f}  ({eff_label(r_visit['h'])})")],
    "", 0.05)
primary_visit.write_html(os.path.join(OUT_DIR, "nb02_card_primary_visit.html"), **PLOTLY_KW)
primary_visit.show()

primary_conv = detail_card(
    "PRIMARY COMPARISON", "Any Email vs No Email — Conversion Rate",
    [("Sample sizes",            f"Control n={r_conv['n2']:,} | Email n={r_conv['n1']:,}"),
     ("Conversion — Control",    f"{r_conv['p2']:.4f}  ({r_conv['x2']:,}/{r_conv['n2']:,})"),
     ("Conversion — Email",      f"{r_conv['p1']:.4f}  ({r_conv['x1']:,}/{r_conv['n1']:,})"),
     ("Difference",              f"{r_conv['diff']:+.4f}"),
     ("Test",                    "Two-Proportion Z-Test (two-tailed)"),
     ("Hypotheses",              "H₀: conv_email = conv_ctrl   H₁: conv_email ≠ conv_ctrl"),
     ("Z-statistic",             f"{r_conv['z']:.4f}"),
     ("P-value",                 f"{r_conv['p']:.2e}"),
     ("95% CI for difference",   f"[{r_conv['ci'][0]:+.4f}, {r_conv['ci'][1]:+.4f}]"),
     ("Alpha (α)",               "0.05"),
     ("Conclusion",               ("✓ SIGNIFICANT — reject H₀" if r_conv["p"]<0.05 else "✗ not significant")),
     ("Effect size (Cohen's h)", f"{r_conv['h']:+.4f}  ({eff_label(r_conv['h'])})")],
    "", 0.05)
primary_conv.write_html(os.path.join(OUT_DIR, "nb02_card_primary_conversion.html"), **PLOTLY_KW)
primary_conv.show()

primary_spend = detail_card(
    "PRIMARY COMPARISON", "Any Email vs No Email — Spending (Welch t-test)",
    [("Sample sizes",            f"Control n={r_spend['n2']:,} | Email n={r_spend['n1']:,}"),
     ("Mean spend — Control",    f"${r_spend['m2']:.4f}"),
     ("Mean spend — Email",      f"${r_spend['m1']:.4f}"),
     ("Difference",              f"${r_spend['diff']:+.4f}"),
     ("Test",                    "Welch's t-test (unequal variances)"),
     ("Hypotheses",              "H₀: μ_email = μ_ctrl   H₁: μ_email ≠ μ_ctrl"),
     ("t-statistic",             f"{r_spend['t']:.4f}"),
     ("P-value",                 f"{r_spend['p']:.2e}"),
     ("Alpha (α)",               "0.05"),
     ("Conclusion",               ("✓ SIGNIFICANT — reject H₀" if r_spend["p"]<0.05 else "✗ not significant")),
     ("Effect size (Cohen's d)", f"{r_spend['d']:+.4f}  ({eff_label(r_spend['d'])})")],
    "", 0.05)
primary_spend.write_html(os.path.join(OUT_DIR, "nb02_card_primary_spend.html"), **PLOTLY_KW)
primary_spend.show()

# ------- Secondary cards (Bonferroni α = 0.0125) -------
alpha_bonf = 0.05 / 4
for arm_name, arm_df in [("Mens E-Mail", mens_g), ("Womens E-Mail", womens_g)]:
    for outcome in ["visit", "conversion"]:
        r = ztest(len(arm_df), arm_df[outcome].sum(),
                  len(control), control[outcome].sum())
        card = detail_card(
            "SECONDARY COMPARISON", f"{arm_name} vs Control — {outcome.capitalize()} Rate",
            [("Sample sizes",            f"Control n={r['n2']:,} | {arm_name} n={r['n1']:,}"),
             (f"{outcome.capitalize()} — Control", f"{r['p2']:.4f}  ({r['x2']:,}/{r['n2']:,})"),
             (f"{outcome.capitalize()} — {arm_name}",  f"{r['p1']:.4f}  ({r['x1']:,}/{r['n1']:,})"),
             ("Difference",              f"{r['diff']:+.4f}"),
             ("Test",                    "Two-Proportion Z-Test (two-tailed)"),
             ("Z-statistic",             f"{r['z']:.4f}"),
             ("P-value",                 f"{r['p']:.2e}"),
             ("95% CI for difference",   f"[{r['ci'][0]:+.4f}, {r['ci'][1]:+.4f}]"),
             ("Bonferroni α (0.05/4)",   f"{alpha_bonf:.4f}"),
             ("Conclusion",               ("✓ SIGNIFICANT — reject H₀" if r["p"]<alpha_bonf else "✗ not significant at Bonferroni α")),
             ("Effect size (Cohen's h)", f"{r['h']:+.4f}  ({eff_label(r['h'])})")],
            "", alpha_bonf)
        fname = f"nb02_card_secondary_{arm_name.split()[0].lower()}_{outcome}.html"
        card.write_html(os.path.join(OUT_DIR, fname), **PLOTLY_KW)
        card.show()

# ------- Comprehensive summary (Primary + Secondary + Exploratory) -------
match_simple = df_blog.groupby("email_match_simple")
match_grp    = {k: df_blog[df_blog["email_match_simple"]==k] for k in ["Match","Mismatch","Mixed","Control"]}
ctl = match_grp["Control"]

summary_rows = []
# Primary
summary_rows.append(("PRIMARY",   "Visit",      "Any Email vs Control",
                     r_visit["p2"], r_visit["p1"], r_visit["diff"], r_visit["z"], r_visit["p"], 0.05, r_visit["h"]))
summary_rows.append(("PRIMARY",   "Conversion", "Any Email vs Control",
                     r_conv["p2"], r_conv["p1"], r_conv["diff"], r_conv["z"], r_conv["p"], 0.05, r_conv["h"]))
summary_rows.append(("PRIMARY",   "Spending",   "Any Email vs Control",
                     r_spend["m2"], r_spend["m1"], r_spend["diff"], r_spend["t"], r_spend["p"], 0.05, r_spend["d"]))
# Secondary
for arm_name, arm_df in [("Mens E-Mail", mens_g), ("Womens E-Mail", womens_g)]:
    for outcome in ["visit", "conversion"]:
        r = ztest(len(arm_df), arm_df[outcome].sum(), len(control), control[outcome].sum())
        summary_rows.append(("SECONDARY", outcome.capitalize(), f"{arm_name} vs Control",
                             r["p2"], r["p1"], r["diff"], r["z"], r["p"], alpha_bonf, r["h"]))
    w = welch(arm_df["spend"], control["spend"])
    summary_rows.append(("SECONDARY", "Spending", f"{arm_name} vs Control",
                         w["m2"], w["m1"], w["diff"], w["t"], w["p"], alpha_bonf, w["d"]))
# Exploratory
for name in ["Match", "Mismatch", "Mixed"]:
    g = match_grp[name]
    for outcome in ["visit", "conversion"]:
        r = ztest(len(g), g[outcome].sum(), len(ctl), ctl[outcome].sum())
        summary_rows.append(("EXPLORATORY", outcome.capitalize(), f"{name} vs Control",
                             r["p2"], r["p1"], r["diff"], r["z"], r["p"], None, r["h"]))
    w = welch(g["spend"], ctl["spend"])
    summary_rows.append(("EXPLORATORY", "Spending", f"{name} vs Control",
                         w["m2"], w["m1"], w["diff"], w["t"], w["p"], None, w["d"]))

def fmt_rate(v, is_spend):
    return f"${v:.4f}" if is_spend else f"{v:.4f}"
def fmt_diff(v, is_spend):
    return f"${v:+.4f}" if is_spend else f"{v:+.4f}"

cols = [[], [], [], [], [], [], [], [], [], [], []]
fill_sig = []
for lvl, outcome, comp, c_rate, t_rate, diff, stat, pval, thr, eff in summary_rows:
    is_spend = outcome == "Spending"
    if thr is None:
        sig = "Yes" if pval < 0.05 else "No"
        thr_txt = "—"
    else:
        sig = "Yes" if pval < thr else "No"
        thr_txt = f"α = {thr:.4f}"
    cols[0].append(lvl); cols[1].append(outcome); cols[2].append(comp)
    cols[3].append(fmt_rate(c_rate, is_spend)); cols[4].append(fmt_rate(t_rate, is_spend))
    cols[5].append(fmt_diff(diff, is_spend))
    cols[6].append(f"{stat:.4f}")
    cols[7].append(f"{pval:.2e}")
    cols[8].append(thr_txt); cols[9].append(f"{eff:+.4f}"); cols[10].append(sig)
    fill_sig.append("#D5F5E3" if sig=="Yes" else "#FADBD8")

header = ["Level", "Outcome", "Comparison", "Control", "Treatment", "Difference",
          "Statistic", "P-value", "Threshold", "Effect size", "Significant?"]
stripe = ["#F8F9F9" if i%2==0 else "white" for i in range(len(cols[0]))]
fill = [stripe]*10 + [fill_sig]

fig = go.Figure(data=[go.Table(
    columnwidth=[90, 90, 220, 90, 100, 100, 90, 110, 100, 100, 110],
    header=dict(values=[f"<b>{h}</b>" for h in header],
                fill_color="#2C3E50", font=dict(color="white", size=13),
                align="center", height=36),
    cells=dict(values=cols, fill_color=fill,
               align=["center","center","left","right","right","right","right","right","center","right","center"],
               font=dict(size=12), height=28))])
fig.update_layout(**{**BASE_LAYOUT, "margin": dict(l=20, r=20, t=60, b=20)},
                  title="Comprehensive Results Summary — Primary, Secondary & Exploratory",
                  height=36 + 28*len(summary_rows) + 90)
fig.write_html(os.path.join(OUT_DIR, "nb02_results_summary_interactive.html"), **PLOTLY_KW)
fig.show()
print("  ✓ nb02_results_summary_interactive.html and individual card HTMLs")


  ✓ nb02_results_table_interactive.html


In [25]:
# Results Display 2: -log10(p-value) bars with clearly separated threshold lines
# Recompute p-values for consistency
any_email = df_blog[df_blog["segment"] != "No E-Mail"]
control   = df_blog[df_blog["segment"] == "No E-Mail"]
mens_g    = df_blog[df_blog["segment"] == "Mens E-Mail"]
womens_g  = df_blog[df_blog["segment"] == "Womens E-Mail"]

tests = [
    ("Any Email vs No Email — visit",    ztest(len(any_email), any_email["visit"].sum(),
                                                len(control),  control["visit"].sum()), 0.05),
    ("Any Email vs No Email — conversion", ztest(len(any_email), any_email["conversion"].sum(),
                                                  len(control),  control["conversion"].sum()), 0.05),
    ("Mens E-Mail vs Control — visit",    ztest(len(mens_g), mens_g["visit"].sum(),
                                                 len(control), control["visit"].sum()), 0.0125),
    ("Womens E-Mail vs Control — visit",  ztest(len(womens_g), womens_g["visit"].sum(),
                                                 len(control), control["visit"].sum()), 0.0125),
    ("Mens E-Mail vs Control — conversion",  ztest(len(mens_g), mens_g["conversion"].sum(),
                                                    len(control), control["conversion"].sum()), 0.0125),
    ("Womens E-Mail vs Control — conversion",ztest(len(womens_g), womens_g["conversion"].sum(),
                                                    len(control), control["conversion"].sum()), 0.0125),
    ("Any Email vs No Email — spend (Welch t)", welch(any_email["spend"], control["spend"]), 0.05),
]

names, neglog_p, thresholds, colors, labels = [], [], [], [], []
CAP = 30.0
for name, r, thr in tests:
    p = r["p"]
    nl = -np.log10(max(p, 1e-300))
    nl_disp = min(nl, CAP)
    names.append(name); thresholds.append(thr); neglog_p.append(nl_disp)
    sig = p < thr
    colors.append("#2ECC71" if sig else "#E74C3C")
    labels.append(f"p = {p:.2e}" if p < 1e-3 else f"p = {p:.4f}")

fig = go.Figure(go.Bar(
    x=neglog_p, y=names, orientation="h",
    marker_color=colors, marker_line=dict(color="black", width=1),
    text=labels, textposition="outside",
    hovertemplate="<b>%{y}</b><br>-log10(p) = %{x:.2f}<extra></extra>",
))

# Threshold vertical lines — use different colors, no conflicting labels on the plot
x_alpha = -np.log10(0.05)     # ~1.30
x_bonf  = -np.log10(0.0125)   # ~1.90

fig.add_shape(type="line", x0=x_alpha, x1=x_alpha, y0=-0.5, y1=len(names)-0.5,
              line=dict(color="#7F8C8D", width=2, dash="dash"))
fig.add_shape(type="line", x0=x_bonf, x1=x_bonf, y0=-0.5, y1=len(names)-0.5,
              line=dict(color="#8E44AD", width=2, dash="dash"))

# Place threshold labels at the TOP, separated horizontally
fig.add_annotation(x=x_alpha, y=1.02, xref="x", yref="paper",
                   text="α = 0.05", showarrow=False,
                   font=dict(color="#7F8C8D", size=11),
                   xanchor="right", yanchor="bottom")
fig.add_annotation(x=x_bonf, y=1.08, xref="x", yref="paper",
                   text="Bonferroni α/4 = 0.0125", showarrow=False,
                   font=dict(color="#8E44AD", size=11),
                   xanchor="left", yanchor="bottom")

fig.update_xaxes(title="-log₁₀(p-value) — higher = stronger evidence",
                 automargin=True, range=[0, CAP * 1.08])
fig.update_yaxes(automargin=True, autorange="reversed")
fig.update_layout(**{**BASE_LAYOUT, "margin": dict(l=280, r=80, t=120, b=80)},
                  title=f"Evidence Strength Across Tests (bars capped at -log10(p) = {CAP:.0f})",
                  height=520, showlegend=False)
fig.write_html(os.path.join(OUT_DIR, "nb02_pvalue_bars_interactive.html"), **PLOTLY_KW)
fig.show()
print("  ✓ nb02_pvalue_bars_interactive.html")


  ✓ nb02_pvalue_bars_interactive.html


In [26]:
# Results Display 3: Effect sizes with interpretation bands
# For proportion tests we use Cohen's h; for the t-test we use Cohen's d.
# Cohen's h magnitude cutoffs (absolute): 0.2 small, 0.5 medium, 0.8 large
# Same cutoffs apply to Cohen's d.
effect_labels = labels
effect_vals = np.array([abs(t[7]) for t in tests])

def band(v):
    if v < 0.2: return "negligible"
    if v < 0.5: return "small"
    if v < 0.8: return "medium"
    return "large"
band_colors = {"negligible":"#D6DBDF","small":"#AED6F1","medium":"#F5B041","large":"#E74C3C"}
bar_colors = [band_colors[band(v)] for v in effect_vals]

fig = go.Figure()
# Background bands
x_max = max(1.0, effect_vals.max() * 1.2)
bands_spec = [(0, 0.2, "#FBFCFC"), (0.2, 0.5, "#EAF2F8"),
              (0.5, 0.8, "#FDEBD0"), (0.8, x_max, "#FADBD8")]
for x0, x1, col in bands_spec:
    fig.add_vrect(x0=x0, x1=x1, fillcolor=col, opacity=0.55, layer="below", line_width=0)

fig.add_trace(go.Bar(
    x=effect_vals, y=effect_labels, orientation="h",
    marker_color=bar_colors, marker_line=dict(color="black", width=0.5),
    text=[f"{v:.3f} ({band(v)})" for v in effect_vals], textposition="outside",
    hovertemplate="<b>%{y}</b><br>|effect size|: %{x:.3f}<extra></extra>",
))
# Band separators
for x in [0.2, 0.5, 0.8]:
    fig.add_vline(x=x, line_dash="dot", line_color="#888")

fig.update_layout(
    **{**BASE_LAYOUT, "margin": dict(l=360, r=80, t=100, b=70)},
    title="Effect Size Magnitude — Cohen's h (proportions) / Cohen's d (spend)<br>"
          "<sub>Bands: <0.2 negligible · 0.2–0.5 small · 0.5–0.8 medium · >0.8 large</sub>",
    xaxis=dict(title="Absolute effect size", automargin=True, range=[0, x_max]),
    yaxis=dict(automargin=True, autorange="reversed"),
    height=max(420, 46*len(effect_labels) + 180),
    showlegend=False,
)
fig.write_html(os.path.join(OUT_DIR, "nb02_effect_sizes_interactive.html"), **PLOTLY_KW)
fig.show()
print("  ✓ nb02_effect_sizes_interactive.html")

  ✓ nb02_effect_sizes_interactive.html
